# Preencher datas e escolher filial

In [1]:
# Data de faturamento (a partir de)
data_inicial_saida_entrega = "14/05/2026"

# Data a ser exibida no excel do protocolo de entrega
data_no_protocolo = "18/05/2026"

# Escolher filial ao trocar chaves da requisição. Altera nome do arquivo excel gerado
# filial = "distribuidora"
filial = "comercio"

# Imports

In [2]:
import requests
import pandas as pd
import os
import json
from dotenv import load_dotenv
from openpyxl import load_workbook
from openpyxl.drawing.image import Image

# Request API

In [3]:
url = "https://app.omie.com.br/api/v1/produtos/nfconsultar/"

# Carrega o .env
load_dotenv()
app_key = os.getenv("app_key")
app_secret = os.getenv("app_secret")


payload = {
    "call": "ListarNF",
    "app_key": app_key,
    "app_secret": app_secret,
    "param": [{
  "pagina": 1,
  "registros_por_pagina": 100,
  "ordenar_por": "CODIGO",
  "dSaiEntInicial": data_inicial_saida_entrega
}]
}

response = requests.post(url, json=payload)

if response.status_code != 200:
    print(f"Erro na requisição da página 1")
else:
    data = response.json()

# Extrai os dados das notas fiscais do json e armazena em uma lista de tuplas (cnpj_cpf, numero_nf)
nfs = data.get("nfCadastro", [])
todas_nfs = []
for nf in nfs:
    numero_nf = nf.get('ide', {}).get('nNF').lstrip('0')
    cnpj_cpf = nf.get('nfDestInt', {}).get('cnpj_cpf')
    todas_nfs.append((cnpj_cpf, numero_nf))

print(todas_nfs)

[('12.365.759/0001-57', '9410'), ('12.365.759/0003-19', '9411'), ('02.318.956/0015-67', '9412'), ('24.860.383/0001-36', '9413'), ('24.860.383/0005-60', '9414'), ('12.365.759/0004-08', '9415'), ('12.365.759/0002-38', '9416'), ('02.318.956/0004-04', '9417'), ('02.318.956/0014-86', '9418'), ('02.318.956/0007-57', '9419'), ('04.879.925/0001-05', '9420'), ('10.254.717/0002-02', '9421'), ('34.035.902/0001-85', '9422'), ('34.035.902/0005-09', '9423'), ('34.035.902/0006-90', '9424'), ('02.318.956/0008-38', '9425'), ('02.318.956/0013-03', '9426'), ('02.318.956/0003-23', '9427'), ('02.318.956/0005-95', '9428'), ('10.254.717/0008-90', '9429'), ('10.254.717/0004-66', '9430'), ('10.254.717/0005-47', '9431'), ('34.035.902/0002-66', '9432'), ('34.035.902/0003-47', '9433'), ('12.365.759/0003-19', '9434'), ('02.318.956/0015-67', '9435'), ('02.318.956/0011-33', '9436'), ('02.318.956/0001-61', '9437'), ('12.365.759/0004-08', '9438'), ('12.365.759/0002-38', '9439'), ('02.318.956/0020-24', '9440'), ('02.31

# Algoritmo Nome Fantasia

In [4]:
#importa "clientes_concatenados_atualizada.csv" para variável df_concatenated
df_concatenated = pd.read_csv("clientes_concatenados_atualizada.csv")

#cria cnpj_col para verificar qual coluna de CNPJ existe
cnpj_col = 'cnpj_cpf' if 'cnpj_cpf' in df_concatenated.columns else 'CNPJ'

# substitui os CNPJs de todas_nfs pelos nomes fantasia usando o df_concatenated atualizado
nome_fantasia_col = 'nome_fantasia' if 'nome_fantasia' in df_concatenated.columns else 'Nome Fantasia'
cnpj_to_nome = dict(zip(df_concatenated[cnpj_col], df_concatenated[nome_fantasia_col]))
todas_nfs = [(cnpj_to_nome.get(cnpj, cnpj), num) for cnpj, num in todas_nfs]

print(f"Substituição com df_concatenated atualizado concluída. Total de NFs: {len(todas_nfs)}")
print(todas_nfs)


Substituição com df_concatenated atualizado concluída. Total de NFs: 48
[('LC6', '9410'), ('LC3', '9411'), ('PIRA VL MARIANA', '9412'), ('MANDA BRASA DEBETTI', '9413'), ('DEBETTI IBIRAPUERA', '9414'), ('LC4', '9415'), ('LC2', '9416'), ('PIRA MORUMBI', '9417'), ('PIRA CENTER NORTE', '9418'), ('PIRA PAULISTA', '9419'), ('ICI BISTRÔ', '9420'), ('ICI BELA CINTRA', '9421'), ('ASTOR OF', '9422'), ('ASTOR CETENCO', '9423'), ('ASTOR VL NOVA CONCEICAO', '9424'), ('PIRA ELDORADO', '9425'), ('PIRA TAMBORE', '9426'), ('PIRA ALPHAVILLE', '9427'), ('PIRA VL LOBOS', '9428'), ('ICI ALPHAVILLE', '9429'), ('ICI VL LOBOS', '9430'), ('ICI JK', '9431'), ('ASTOR JK', '9432'), ('ASTOR SP', '9433'), ('LC3', '9434'), ('PIRA VL MARIANA', '9435'), ('PIRA PRAINHA', '9436'), ('PIRA PINHEIRO', '9437'), ('LC4', '9438'), ('LC2', '9439'), ('PIRA CIDADE SP', '9440'), ('PIRA MORUMBI', '9441'), ('PIRA CENTER NORTE', '9442'), ('PIRA PAULISTA', '9443'), ('ICI BISTRÔ', '9444'), ('ICI BELA CINTRA', '9445'), ('ASTOR OF', '944

# Algoritmo Planilhas

In [5]:
# Configuração de blocos
blocos_por_planilha = 5
linhas_por_bloco = 13
linha_inicio = 7

workbook_index = 1
block_index = 0

wb = load_workbook("protocolo_entrega_alterado.xlsx")
ws = wb.active

for i in range(0, len(todas_nfs), 2):
    # Inicia um novo workbook a cada blocos_por_planilha blocos
    if block_index >= blocos_por_planilha:
        file_name = f"{filial}_protocolo_entrega_parte_{workbook_index}.xlsx"

        celulas = ['A1', 'A14', 'A27', 'A40', 'A53', 'I1', 'I14', 'I27', 'I40', 'I53']
        for celula in celulas:
            nova_img = Image('logo_v2.png')
            nova_img.width = 115
            nova_img.height = 110
            ws.add_image(nova_img, celula)

        wb.save(f"C:\\projetos\\protocolo_nfs\\output\\{file_name}")
        print(f"Salvou {file_name} com {block_index} blocos")

        workbook_index += 1
        block_index = 0
        wb = load_workbook("protocolo_entrega_alterado.xlsx")
        ws = wb.active

    lote = todas_nfs[i:i+2]
    print(f"Processando lote {block_index+1} da planilha {workbook_index}: {lote}")

    row_base = linha_inicio + block_index * linhas_por_bloco
    print(f"Base row para este bloco: {row_base}")

    if len(lote) == 2:
        cnpj1, num1 = lote[0]
        cnpj2, num2 = lote[1]

        # Escreve CNPJs em pares na mesma linha
        ws.cell(row=row_base, column=3, value=cnpj1)   # Coluna C
        ws.cell(row=row_base, column=11, value=cnpj2)  # Coluna K

        # Escreve data_no_protocolo em pares na linha seguinte
        ws.cell(row=row_base + 1, column=3, value=data_no_protocolo)   # Coluna C
        ws.cell(row=row_base + 1, column=11, value=data_no_protocolo)  # Coluna K

        # Escreve números em pares na linha seguinte
        ws.cell(row=row_base + 3, column=3, value=num1)  # Coluna C
        ws.cell(row=row_base + 3, column=11, value=num2) # Coluna K

        print(f"Bloco {block_index+1} planilha {workbook_index}: row {row_base}/{row_base+3} -> {cnpj1},{cnpj2}/{num1},{num2}")

    elif len(lote) == 1:
        cnpj1, num1 = lote[0]

        ws.cell(row=row_base, column=3, value=cnpj1)   # Coluna C
        ws.cell(row=row_base + 1, column=3, value=data_no_protocolo) # Coluna C
        ws.cell(row=row_base + 3, column=3, value=num1) # Coluna C (seguindo o padrão de par)
        
       

        print(f"Bloco {block_index+1} planilha {workbook_index} (solitário): row {row_base}/{row_base+3} -> {cnpj1}/{num1}")

    else:
        continue

    block_index += 1

# Salva o último workbook em aberto
if block_index > 0:
    file_name = f"{filial}_protocolo_entrega_parte_{workbook_index}.xlsx"
    
    celulas = ['A1', 'A14', 'A27', 'A40', 'A53', 'I1', 'I14', 'I27', 'I40', 'I53']
    for celula in celulas:
        nova_img = Image('logo_v2.png')
        nova_img.width = 115
        nova_img.height = 110
        ws.add_image(nova_img, celula)

    wb.save(f"C:\\projetos\\protocolo_nfs\\output\\{file_name}")
    print(f"Salvou {file_name} com {block_index} blocos")

Processando lote 1 da planilha 1: [('LC6', '9410'), ('LC3', '9411')]
Base row para este bloco: 7
Bloco 1 planilha 1: row 7/10 -> LC6,LC3/9410,9411
Processando lote 2 da planilha 1: [('PIRA VL MARIANA', '9412'), ('MANDA BRASA DEBETTI', '9413')]
Base row para este bloco: 20
Bloco 2 planilha 1: row 20/23 -> PIRA VL MARIANA,MANDA BRASA DEBETTI/9412,9413
Processando lote 3 da planilha 1: [('DEBETTI IBIRAPUERA', '9414'), ('LC4', '9415')]
Base row para este bloco: 33
Bloco 3 planilha 1: row 33/36 -> DEBETTI IBIRAPUERA,LC4/9414,9415
Processando lote 4 da planilha 1: [('LC2', '9416'), ('PIRA MORUMBI', '9417')]
Base row para este bloco: 46
Bloco 4 planilha 1: row 46/49 -> LC2,PIRA MORUMBI/9416,9417
Processando lote 5 da planilha 1: [('PIRA CENTER NORTE', '9418'), ('PIRA PAULISTA', '9419')]
Base row para este bloco: 59
Bloco 5 planilha 1: row 59/62 -> PIRA CENTER NORTE,PIRA PAULISTA/9418,9419
Salvou comercio_protocolo_entrega_parte_1.xlsx com 5 blocos
Processando lote 1 da planilha 2: [('ICI BIST